# Module 11: Evaluation & Benchmarking
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jingyaogong/minimind-colab/blob/master/notebooks/11_evaluation.ipynb)

## Overview
How do we know if our model actually improved?  
This module covers the full evaluation pipeline — from quick qualitative checks to formal benchmarks.

### Evaluation Taxonomy

| Type | Method | Speed | Objectivity |
|---|---|---|---|
| Qualitative | Side-by-side comparison | Fast | Low |
| Intrinsic | Perplexity on held-out text | Fast | Medium |
| Capability | Benchmark suites (C-Eval, CMMLU) | Slow | High |
| Task-specific | Tool-call accuracy, ROUGE, BLEU | Medium | High |

### Key Chinese LLM Benchmarks
- **C-Eval** — 52 subjects, multiple-choice, covering STEM and humanities
- **CMMLU** — 67 subjects, Chinese Massive Multitask Language Understanding
- **MMLU** — English equivalent, for cross-lingual comparison

We will use [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) for formal evaluation.


In [ ]:
# ── Setup ──
import subprocess, sys, os

if not os.path.exists('/content/minimind-colab'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/jingyaogong/minimind-colab.git',
                    '/content/minimind-colab'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                '/content/minimind-colab/requirements.txt'], check=True)

sys.path.insert(0, '/content/minimind-colab')
print("✅ Setup complete")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo "CPU only"


## 🔄 Convert Model to Transformers Format

Before using `lm-evaluation-harness` or deploying via the OpenAI-compatible API,  
we need to save the checkpoint in HuggingFace Transformers format  
(using `scripts/convert_model.py` → `convert_torch2transformers`).


In [ ]:
import sys, os, json, torch, warnings
sys.path.insert(0, '/content/minimind-colab')
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM


def convert_to_transformers(weight_name, output_dir, hidden_size=768, num_layers=8):
    """Convert a .pth checkpoint to HuggingFace Transformers format."""
    MiniMindConfig.register_for_auto_class()
    MiniMindForCausalLM.register_for_auto_class("AutoModelForCausalLM")

    lm_config = MiniMindConfig(hidden_size=hidden_size, num_hidden_layers=num_layers)
    model     = MiniMindForCausalLM(lm_config)

    ckp_path = f'/content/minimind-colab/out/{weight_name}_{hidden_size}.pth'
    if not os.path.exists(ckp_path):
        print(f"❌ Checkpoint not found: {ckp_path}")
        return None

    state_dict = torch.load(ckp_path, map_location='cpu')
    model.load_state_dict(state_dict, strict=False)
    model = model.to(torch.float16)

    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir, safe_serialization=False)

    tokenizer = AutoTokenizer.from_pretrained('/content/minimind-colab/model')
    tokenizer.save_pretrained(output_dir)

    # Patch config for Transformers >=5.0 compatibility
    config_path = os.path.join(output_dir, "config.json")
    with open(config_path) as fh:
        config = json.load(fh)
    config['rope_theta']   = lm_config.rope_theta
    config['rope_scaling'] = None
    config.pop('rope_parameters', None)
    with open(config_path, 'w') as fh:
        json.dump(config, fh, indent=2, ensure_ascii=False)

    print(f"✅ Saved → {output_dir}")
    return output_dir


# Convert the first available checkpoint
converted_path = None
for weight in ['grpo', 'dpo', 'full_sft', 'pretrain']:
    ckp = f'/content/minimind-colab/out/{weight}_768.pth'
    if os.path.exists(ckp):
        out_dir = f'/content/minimind_eval/{weight}'
        converted_path = convert_to_transformers(weight, out_dir)
        print(f"Converted '{weight}' → {out_dir}")
        break
else:
    print("No local checkpoints found.  Using pre-trained model for evaluation.")


## 📊 Qualitative Evaluation: Side-by-Side Comparison


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
models_to_compare = {}

checkpoints = {
    'pretrain': '/content/minimind-colab/out/pretrain_768.pth',
    'sft':      '/content/minimind-colab/out/full_sft_768.pth',
    'dpo':      '/content/minimind-colab/out/dpo_768.pth',
    'grpo':     '/content/minimind-colab/out/grpo_768.pth',
}

lm_config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
for name, ckp in checkpoints.items():
    if os.path.exists(ckp):
        m = MiniMindForCausalLM(lm_config)
        m.load_state_dict(torch.load(ckp, map_location=device), strict=False)
        models_to_compare[name] = m.half().eval().to(device)
        print(f"✅ Loaded: {name}")

tokenizer = AutoTokenizer.from_pretrained('/content/minimind-colab/model')

# Fall back to pre-trained HF model if no local checkpoints exist
if not models_to_compare:
    tokenizer = AutoTokenizer.from_pretrained('/content/minimind-3', trust_remote_code=True)
    _m = AutoModelForCausalLM.from_pretrained('/content/minimind-3', trust_remote_code=True)
    models_to_compare['pretrained_minimind3'] = _m.half().eval().to(device)
    print("✅ Using pre-trained MiniMind-3 (no local checkpoints found)")

print(f"\nModels ready for comparison: {list(models_to_compare.keys())}")


In [ ]:
def evaluate_model(model, tok, prompt, is_pretrain=False, max_tokens=150):
    if is_pretrain:
        inp_text = tok.bos_token + prompt
    else:
        conv     = [{"role": "user", "content": prompt}]
        inp_text = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)

    inp = tok(inp_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            inp["input_ids"], attention_mask=inp["attention_mask"],
            max_new_tokens=max_tokens, do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id
        )
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)


eval_prompts = [
    "解释量子力学的基本原理",
    "写一首关于春天的短诗",
    "如何学好编程？给出3个建议",
]

print("=" * 80)
print("SIDE-BY-SIDE MODEL COMPARISON")
print("=" * 80)
for prompt in eval_prompts:
    print(f"\n💬 Prompt: {prompt}\n")
    for name, m in models_to_compare.items():
        is_pretrain = (name == 'pretrain')
        response = evaluate_model(m, tokenizer, prompt, is_pretrain=is_pretrain)
        print(f"  [{name.upper():15}]: {response[:200]}")
        print("  " + "-" * 60)
    print()


## 📏 Quantitative Metrics: Perplexity
**Perplexity** (PPL) measures how well the model predicts a held-out test corpus.  
Lower is better.  PPL is calculated as `exp(cross-entropy loss)`.


In [ ]:
import math

def compute_perplexity(model, tok, texts, batch_size=4, max_len=256):
    """Compute average perplexity on a list of texts."""
    model.eval()
    total_loss, total_tokens = 0.0, 0

    for text in texts[:20]:   # limit for speed
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=max_len).to(device)
        input_ids = inputs["input_ids"]
        if input_ids.shape[1] < 2:
            continue
        with torch.no_grad():
            out  = model(input_ids, labels=input_ids)
            n    = input_ids.shape[1] - 1
            total_loss   += out.loss.item() * n
            total_tokens += n

    return math.exp(total_loss / max(total_tokens, 1))


# Load test samples
test_texts = []
try:
    with open('/content/minimind-colab/dataset/sft_t2t_mini.jsonl') as f:
        for i, line in enumerate(f):
            if i >= 20:
                break
            data = json.loads(line)
            if 'conversations' in data:
                test_texts.append(' '.join(t['content'] for t in data['conversations']))
except FileNotFoundError:
    test_texts = ["测试文本示例，用于评估模型困惑度。这是一个较长的句子以确保有足够的词元。"]

print("Perplexity Comparison (lower is better):")
print("-" * 45)
for name, m in models_to_compare.items():
    ppl = compute_perplexity(m, tokenizer, test_texts)
    print(f"  {name:20}: PPL = {ppl:8.2f}")


## 🏆 Formal Benchmarking with lm-evaluation-harness


In [ ]:
# Install lm-evaluation-harness
!pip install -q lm-eval

print("Full benchmark evaluation (30-60 min).  Example commands:\n")
print("""
# Single benchmark — C-Eval:
!lm_eval \\
    --model hf \\
    --model_args pretrained=/content/minimind_eval/full_sft \\
    --tasks ceval-valid \\
    --batch_size 8 \\
    --device cuda \\
    --output_path /content/minimind_eval/results/

# Multiple benchmarks at once:
!lm_eval \\
    --model hf \\
    --model_args pretrained=/content/minimind_eval/full_sft \\
    --tasks ceval-valid,cmmlu \\
    --batch_size 8 \\
    --device cuda \\
    --output_path /content/minimind_eval/results/
""")

# Quick sanity-check run on PIQA (fast, ~2 min)
import os
if converted_path and os.path.exists(converted_path):
    print(f"Running quick evaluation on PIQA (100 samples) using: {converted_path}")
    !lm_eval --model hf --model_args pretrained={converted_path} \
             --tasks piqa --batch_size 8 --device cuda --limit 100
else:
    print("(Skipped — no converted model found. Run the conversion cell above first.)")


## 📈 Summary: Training Pipeline Results


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

stages    = ['Random Init', 'After Pretrain', 'After SFT', 'After DPO', 'After GRPO']
ppls_plot = [100, 12.5, 8.2, 7.8, 7.1]   # approximate expected PPL for MiniMind 768-hidden model (~85M params); idealized reference values
colors    = ['#9E9E9E', '#2196F3', '#4CAF50', '#FF9800', '#F44336']

cap_labels  = ['Word Completion', 'Instruction Follow', 'Value Aligned', 'RL Improved']
cap_matrix  = np.array([
    [0, 1, 1, 1, 1],
    [0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 0, 1],
], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Perplexity bar chart ──
bars = axes[0].bar(range(len(stages)), ppls_plot, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_xticks(range(len(stages)))
axes[0].set_xticklabels(stages, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Perplexity (lower is better)', fontsize=10)
axes[0].set_title('Model Perplexity Across Training Stages', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, 115)
axes[0].grid(axis='y', alpha=0.3)
for i, (bar, ppl) in enumerate(zip(bars, ppls_plot)):
    label = f'{ppl:.1f}' if i > 0 else '~∞'
    axes[0].text(bar.get_x() + bar.get_width()/2, ppl + 2, label,
                 ha='center', fontsize=9, fontweight='bold')

# ── Capability heatmap ──
im = axes[1].imshow(cap_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1].set_xticks(range(len(stages)))
axes[1].set_xticklabels(stages, rotation=20, ha='right', fontsize=9)
axes[1].set_yticks(range(len(cap_labels)))
axes[1].set_yticklabels(cap_labels, fontsize=9)
axes[1].set_title('Capability Matrix', fontsize=11, fontweight='bold')
for i in range(len(cap_labels)):
    for j in range(len(stages)):
        axes[1].text(j, i, '✓' if cap_matrix[i, j] else '✗',
                     ha='center', va='center', fontsize=14,
                     color='white' if cap_matrix[i, j] else '#555')

plt.tight_layout()
out_img = '/content/minimind-colab/training_summary.png'
plt.savefig(out_img, dpi=100, bbox_inches='tight')
plt.show()
print(f"Summary chart saved: {out_img}")


## 🌐 Deploy: OpenAI-Compatible API (Optional)

MiniMind ships with `scripts/serve_openai_api.py` which exposes a  
`POST /v1/chat/completions` endpoint compatible with the OpenAI SDK.  
Combine it with **ngrok** to get a public URL from Colab.


In [ ]:
print("""
Steps to deploy the model as a public API from Colab:

1. Install dependencies:
   !pip install pyngrok flask

2. Convert checkpoint to Transformers format (see cell above), then:

3. Start the API server:
   import subprocess
   server = subprocess.Popen([
       'python', '/content/minimind-colab/scripts/serve_openai_api.py',
       '--load_from', '/content/minimind_eval/full_sft',
       '--port', '8888'
   ])

4. Open ngrok tunnel:
   from pyngrok import ngrok
   ngrok.set_auth_token("YOUR_NGROK_TOKEN")   # free at ngrok.com
   tunnel = ngrok.connect(8888)
   print(f"API URL: {tunnel.public_url}")

5. Call it with curl:
   curl -X POST {tunnel.public_url}/v1/chat/completions \\
        -H 'Content-Type: application/json' \\
        -d '{
              "model": "minimind",
              "messages": [{"role": "user", "content": "你好，介绍一下自己"}],
              "temperature": 0.7
            }'

   Or with the OpenAI Python SDK:
   from openai import OpenAI
   client = OpenAI(base_url=f"{tunnel.public_url}/v1", api_key="any")
   resp = client.chat.completions.create(
       model="minimind",
       messages=[{"role": "user", "content": "写一首关于AI的诗"}]
   )
   print(resp.choices[0].message.content)
""")


In [ ]:
# Memory cleanup
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free  = torch.cuda.memory_reserved() - torch.cuda.memory_allocated()
    print(f"GPU memory freed.  Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("CPU environment — nothing to free")


## 🎓 Course Wrap-Up

Congratulations! 🎉 You have completed the **MiniMind LLM Training Course**.

### What You Built
| Module | Topic | Output |
|---|---|---|
| 01–02 | Setup, Tokenizer, Data | Working Colab environment |
| 03 | Model Architecture | Transformer deep-dive |
| 04 | Pretraining | Language modelling from scratch |
| 05 | SFT | Instruction-following model |
| 06 | LoRA | Parameter-efficient fine-tuning |
| 07 | Distillation | Smaller model from larger teacher |
| 08 | DPO Alignment | Preference-optimised model |
| 09 | GRPO | RL-improved model |
| 10 | Tool Calling | Agentic model |
| 11 | Evaluation | Benchmarked model |

### Next Steps
- 🔬 Run full C-Eval / CMMLU benchmarks on your best checkpoint
- 🚀 Deploy your model via the OpenAI-compatible API
- 📊 Share your training curves on GitHub Discussions
- 🤝 Contribute improvements back to [minimind-colab](https://github.com/jingyaogong/minimind-colab)


## 📝 Final Student Project

Design and implement a **mini evaluation suite** for MiniMind:

### Requirements
1. **Dataset** — Curate 50 Chinese multiple-choice questions (10 per subject):  
   math, science, history, language, coding.

2. **Evaluation script** — Score each of the training checkpoints  
   (`pretrain`, `sft`, `dpo`, `grpo`) on your dataset.

3. **Analysis** — Create a table and chart showing accuracy per subject per stage.

4. **Report** — Write a 1-page analysis answering:  
   - Which training stage improved which subjects most?  
   - Did GRPO help beyond DPO?  
   - What are the failure modes?

### Evaluation Criteria
| Criterion | Weight |
|---|---|
| Correctness of evaluation logic | 30% |
| Quality of dataset (diversity, difficulty) | 20% |
| Depth of analysis | 30% |
| Code clarity and reproducibility | 20% |
